source API URL : "https://geocoding-api.open-meteo.com/v1/search?name=kovilpatti&count=10&language=en&format=json"

JSON Target File Path : "abfss://bronze@datalakestorageaccountname.dfs.core.windows.net/geo-location/"

In [0]:
#geoLocationSourceAPIURL = "https://geocoding-api.open-meteo.com/v1/search?name=kovilpatti&count=10&language=en&format=json"

geoLocationSourceAPIBaseURL = "https://geocoding-api.open-meteo.com/v1/search?name="
geoLocationSourceAPIURLOptions = "&count=10&language=en&format=json"

geoLocationSinkLayerName = 'bronze'
geoLocationSinkStorageAccountName = 'adldatalakehouseyubai'
geoLocationSinkFolderName = 'geo-location'

geoLocationSinkFolderPath = f"abfss://{geoLocationSinkLayerName}@{geoLocationSinkStorageAccountName}.dfs.core.windows.net/{geoLocationSinkFolderName}"

In [0]:
import requests
import json
import pandas as pds

In [0]:
dailyPricingMarketNamesDF = spark.sql("SELECT MARKET_NAME from pricing_analytics.gold.reporting_dim_market_gold LIMIT 50 ")

In [0]:
import requests
import json

marketNames = [
    row["MARKET_NAME"]
    for row in dailyPricingMarketNamesDF.collect()
]

geoLocationAPIResponseList = []

for marketName in marketNames:
  
    geoLocationSourceAPIURL = (
        f"{geoLocationSourceAPIBaseURL}"
        f"{marketName}"
        f"{geoLocationSourceAPIURLOptions}"
    )

    geoLocationAPIResponse = requests.get(
        geoLocationSourceAPIURL,
        timeout=30
    ).json()

    if isinstance(geoLocationAPIResponse, dict):
        geoLocationAPIResponse["marketName"] = marketName
        geoLocationAPIResponseList.append(
            json.dumps(geoLocationAPIResponse)
        )


tempGeoLocationPath = f"{geoLocationSinkFolderPath}_temp"

dbutils.fs.put(
    tempGeoLocationPath,
    "\n".join(geoLocationAPIResponseList),
    overwrite=True
)

geoLocationSparkDF = spark.read.json(tempGeoLocationPath)

(geoLocationSparkDF
 .filter("results.admin1 IS NOT NULL")
 .write
 .mode("overwrite")
 .json(geoLocationSinkFolderPath))

In [0]:
from pyspark.sql.functions import col, array_contains
geoLocationBronzeDF = (spark
                       .read
                                             .json(geoLocationSinkFolderPath))
                       

display(geoLocationBronzeDF)